In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import text_hammer as th
from tqdm.notebook import tqdm
import gensim.downloader as api

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Bidirectional, Dropout

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn import metrics



In [ ]:
train_path = "/content/train.txt"
test_path  = "/content/test.txt"
val_path   = "/content/val.txt"

df_train = pd.read_csv(train_path, header=None, sep=";", names=["Input","Sentiment"])
df_test  = pd.read_csv(test_path,  header=None, sep=";", names=["Input","Sentiment"])
df_val   = pd.read_csv(val_path,   header=None, sep=";", names=["Input","Sentiment"])

df_train.head()


In [ ]:
sns.countplot(df_train.Sentiment)
plt.title("Class Distribution")
plt.show()


In [ ]:
tqdm.pandas()

def text_preprocessing(df, col_name):
    df[col_name] = df[col_name].progress_apply(lambda x: str(x).lower())
    df[col_name] = df[col_name].progress_apply(lambda x: th.cont_exp(x))
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_emails(x))
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_html_tags(x))
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_special_chars(x))
    df[col_name] = df[col_name].progress_apply(lambda x: th.remove_accented_chars(x))
    df[col_name] = df[col_name].progress_apply(lambda x: th.make_base(x))
    return df


In [ ]:
df_cleaned_train = text_preprocessing(df_train.copy(), "Input")


In [ ]:
df_train.Sentiment.unique()


In [ ]:
label_map = {
    'joy':0,
    'anger':1,
    'love':2,
    'sadness':3,
    'fear':4,
    'surprise':5
}

df_cleaned_train["Sentiment"] = df_cleaned_train["Sentiment"].replace(label_map)
df_test["Sentiment"] = df_test["Sentiment"].replace(label_map)
df_val["Sentiment"] = df_val["Sentiment"].replace(label_map)

y_train = to_categorical(df_cleaned_train.Sentiment.values)
y_test  = to_categorical(df_test.Sentiment.values)
y_val   = to_categorical(df_val.Sentiment.values)


In [ ]:
df_cleaned_train.Sentiment.unique()


In [ ]:
y_train.shape

In [ ]:
num_words = 10000
tokenizer = Tokenizer(num_words=num_words, lower=True)

df_total = pd.concat([df_cleaned_train['Input'], df_test["Input"]], axis=0)
tokenizer.fit_on_texts(df_total)


In [ ]:
max_len = 300

X_train = tokenizer.texts_to_sequences(df_cleaned_train["Input"])
X_test  = tokenizer.texts_to_sequences(df_test["Input"])
X_val   = tokenizer.texts_to_sequences(df_val["Input"])

X_train_pad = pad_sequences(X_train, maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test,  maxlen=max_len, padding='post')
X_val_pad   = pad_sequences(X_val,   maxlen=max_len, padding='post')


In [ ]:
print("Downloading GloVe...")
glove = api.load("glove-wiki-gigaword-100")  # 100-dimension


In [ ]:
vector_size = 100
embedding_matrix = np.zeros((num_words, vector_size))

for word, idx in tokenizer.word_index.items():
    if idx < num_words:
        if word in glove:
            embedding_matrix[idx] = glove[word]
        else:
            embedding_matrix[idx] = np.zeros(vector_size)


In [ ]:
EMBEDDING_DIM = 100
class_num = 6

model = Sequential()

model.add(Embedding(
    input_dim=num_words,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    trainable=False
))

model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(100, return_sequences=True)))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(200, return_sequences=True)))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(100, return_sequences=False)))

model.add(Dense(class_num, activation='softmax'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()


In [ ]:
es = EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='min')
mc = ModelCheckpoint("best_model.h5", monitor='val_accuracy', save_best_only=True, verbose=1, mode='max')


In [ ]:
history = model.fit(
    X_train_pad, y_train,
    epochs=25,
    batch_size=120,
    validation_data=(X_val_pad, y_val),
    callbacks=[es, mc],
    verbose=1
)


In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title("Training History")
plt.show()


In [ ]:
y_pred = np.argmax(model.predict(X_test_pad), axis=1)
y_true = np.argmax(y_test, axis=1)

print(metrics.classification_report(y_true, y_pred))


In [ ]:
label_map = {0:'joy', 1:'anger', 2:'love', 3:'sadness', 4:'fear', 5:'surprise'}


In [ ]:
import numpy as np

def predict_emotion(sentences):
    # Convert to sequences
    seq = tokenizer.texts_to_sequences(sentences)
    seq_pad = pad_sequences(seq, maxlen=max_len, padding='post')

    # Get prediction probabilities
    preds = model.predict(seq_pad)

    for i, p in enumerate(preds):
        print(f"\nINPUT: {sentences[i]}")
        print("Top 2 predictions:")

        # Get top 2 indices sorted by probability
        top_idx = np.argsort(p)[::-1][:2]

        for idx in top_idx:
            print(f"   {label_map[idx]} → {p[idx]:.4f}")


In [ ]:
sentences = [
    "Today is a bad day for me",
    "she always gets angry if she doesn't get her own way",
    "What a beautiful day",
    "That horror movie is so scary",
    "Wow! what a lovely surprise",
    "I am not so happy today",

    # Provided examples:
    "An overwhelming sense of euphoria washed over him as he realized that months of arduous work had finally culminated in this moment of success",  # joy
    "A deep, lingering melancholy settled upon her as she stared out at the rain-streaked window, contemplating the opportunities she had lost.",  # sadness
    "His initial annoyance quickly festered into a simmering rage as he dealt with the bureaucratic incompetence that had delayed his project for weeks.",  # anger
    "A creeping sense of dread began to pervade the room as the silence stretched on, each tick of the clock amplifying the unspoken apprehension",  # fear
    "The board was utterly astounded by the quarterly report, which revealed unprecedented growth in a market everyone had written off.",  # surprise
    "The document outlines the procedural steps required for the submission of the annual financial report, adhering strictly to established guidelines.",  # neutral (model must guess)
]

predict_emotion(sentences)
